In [ ]:
import pandas as pd

# TO DO

- motivation
- about openFDA
- limitations

### RxNorm & RXCUIs

RxNorm is a standardised naming system for medicines in the US, maintained by the National Library of Medicine. The same drug can be described in many ways ("Tylenol Extra Strength", "APAP 500mg tab", "acetaminophen 500 MG Oral Tablet"), and RxNorm gives each distinct concept one canonical name and ID so that different systems (pharmacies, hospitals, the FDA) can refer to the same thing.

An RXCUI (RxNorm Concept Unique Identifier) is that ID. Your label's openfda.rxcui field lists the RxNorm concepts for the products on that label.


## Creating a drug labels dataset

Created `drugs_to_search_subset.csv` containing 5 drugs, just to test the code:

In [ ]:
pd.read_csv("drugs_to_search_subset.csv")

There are 2 unique product types: `HUMAN OTC DRUG` (over the counter) and `HUMAN PRESCRIPTION DRUG`. It's important that the subset contains examples of both, since they contain different sections in the OpenFDA database.

In config.py, I set `products_path` to `drugs_to_search_subset.csv` and ran `find_rxcuis.py`. This asks RxNav for all SCD concepts... writes relevant results to `rxcui_candidates.csv`. For the 5 drugs listed above, hundreds of results were returned. I manually chose the most relevant results (e.g the 'standard strength' version) and copied their rxcuis into the corresponding column in `drugs_to_search_subset.csv`.

I then ran `fetch_drug_labels.csv`, which searched OpenFDA for the exact RXCUIs in `drugs_to_search_subset.csv` and wrote the results to one json file per drug (see the `raw_labels` directory).

I had originally tried creating a list of drug names such as `acetaminophen`, but querying OpenFDA returned hundreds of results for different brands and dosages of the drug. This motivated me finding precise RXCUIs for 'standard strengths'.

### Cleaning the dataset

We now have 5 JSON files which we want to turn into Langchain `Document`s.

Each JSON file contains a lot of information, so we need to decide what to keep.

In [ ]:
import json
from collections import Counter
from pathlib import Path, PosixPath

sample_path = PosixPath("raw_labels/ibuprofen_200mg_tab.json")
label = json.loads(sample_path.read_text())

section_counts, section_chars = Counter(), Counter()
for key, value in label.items():
    print(value)
    # if isinstance(value, list) and value and isinstance(value[0], str):
    #     section_counts[key] += 1
    #     section_chars[key] += sum(len(v) for v in value)

In [ ]:
import json
from collections import Counter
from pathlib import Path, PosixPath

label_dir = Path("raw_labels")
labels = [
    json.loads(p.read_text())
    for p in label_dir.glob("*.json")
    if p.name != "manifest.json"
]

section_counts, section_chars = Counter(), Counter()
for label in labels:
    for key, value in label.items():
        if isinstance(value, list) and value and isinstance(value[0], str):
            section_counts[key] += 1
            section_chars[key] += sum(len(v) for v in value)

for key, n in section_counts.most_common():
    print(f"{key:45} {n:3} labels   avg {section_chars[key] // n:7,} chars")